# QM9 `gap_eV` 予測 — RDKit 特徴量 × LightGBM（提出3本）

配布された `smiles` だけから特徴量を生成し、`gap_eV`（HOMO-LUMO ギャップ, eV）を予測する。

## このノートブックの方針
評価指標は **MAE**。同一の 5-fold 交差検証（`KFold(n_splits=5, shuffle=True, random_state=8)`）で
3つの方針を公平に比較し、テスト予測を提出用CSVとして書き出す。

| 方針 | 特徴量 | モデル |
| --- | --- | --- |
| 方針1 | RDKit 標準2D記述子 | LightGBM |
| 方針2 | RDKit記述子 ＋ Morgan フィンガープリント | LightGBM |
| 方針3 | 方針1・2 の予測ブレンド | （重み最適化） |

## コンペ制約（順守事項）
- 外部データを追加しない（配布SMILESからの特徴量生成のみ）。
- 乱数シードは `8` に統一。
- テストの並び順は変更しない／予測値を手作業で変更しない。
- 提出は最大3モデル。各CSVは `smiles,gap_eV` の列順・4000行。

## 出力
- `submission_1_descriptors_lgbm.csv`
- `submission_2_descriptors_morgan_lgbm.csv`
- `submission_3_blend.csv`


## 1. セットアップ

必要なライブラリと定数（シード・fold数・入出力パス）をまとめて定義する。
入力CSVはこのノートブックと同じフォルダに置いてある前提。

In [1]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")  # RDKitのパース警告を抑制

SEED = 8            # コンペ指定の乱数シード
N_SPLITS = 5        # 5-fold 交差検証
DATA_DIR = Path(".")               # このノートブックと同じフォルダ
TRAIN_PATH = DATA_DIR / "qm9_bandgap_train.csv"
TEST_PATH = DATA_DIR / "qm9_bandgap_test_without_answer.csv"

print("rdkit   :", rdkit.__version__)
print("lightgbm:", lgb.__version__)
print("seed    :", SEED, " / n_splits:", N_SPLITS)


rdkit   : 2026.03.4
lightgbm: 4.7.0
seed    : 8  / n_splits: 5


## 2. データ読み込み

学習データ（`smiles`, `gap_eV`）とテストデータ（`smiles`）を読み込み、
形状・先頭行・目的変数の分布を確認する。

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("train:", train_df.shape, " test:", test_df.shape)
display(train_df.head(3))

print("gap_eV の統計量:")
print(train_df["gap_eV"].describe())


train: (15000, 2)  test: (4000, 1)


,smiles,gap_eV
0,C#CC1(CNC1=O)C#C,7.094012
1,CC1CC=CCOC=N1,6.854552
2,CC1=C2CCC3C(C1)C23,5.839566


gap_eV の統計量:
count    15000.000000
mean         6.823883
std          1.288640
min          1.774183
25%          5.885826
50%          6.781081
75%          7.834162
max         10.718570
Name: gap_eV, dtype: float64


## 3. SMILES → 分子（Mol）変換

すべての `smiles` を `Chem.MolFromSmiles()` で RDKit の Mol に変換する。
変換に失敗した行は「行番号・SMILES」をログ出力する（**行の削除はしない**）。
テストに変換不能なSMILESがあると提出に影響するため、その場合は明示的に停止する。

In [3]:
def smiles_to_mols(smiles_list):
    """SMILESのリストをMolへ変換し、(mols, 失敗した[(行番号, SMILES)]) を返す。"""
    mols, invalid = [], []
    for i, smi in enumerate(smiles_list):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            invalid.append((i, smi))
        mols.append(m)
    return mols, invalid


train_mols, train_invalid = smiles_to_mols(train_df["smiles"].tolist())
test_mols, test_invalid = smiles_to_mols(test_df["smiles"].tolist())

print(f"変換失敗: train={len(train_invalid)} 件, test={len(test_invalid)} 件")
for idx, smi in (train_invalid + test_invalid)[:10]:
    print("  invalid row:", idx, smi)

# テスト側に変換不能があると4000行の予測が作れないため停止する
assert not test_invalid, "テストに変換不能なSMILESがあります"


変換失敗: train=0 件, test=0 件


## 4. 特徴量生成

2種類の特徴量を作る。

1. **RDKit 標準2D記述子**：`Descriptors.CalcMolDescriptors()` で約200個を一括取得。
   分子量・環の数・極性表面積など、化学的に解釈しやすい量。
2. **Morgan フィンガープリント**：`rdFingerprintGenerator.GetMorganGenerator()` による
   count 型（radius=2, 2048bit）。各分子の部分構造の有無・個数をビット列で表す。

（記述子の計算は分子数が多いと時間がかかる。進捗バーで確認できる。）

In [4]:
def calc_descriptors(mols):
    """RDKit標準2D記述子を一括計算してDataFrameで返す。"""
    rows = []
    for m in tqdm(mols, desc="descriptors"):
        rows.append({} if m is None else Descriptors.CalcMolDescriptors(m))
    return pd.DataFrame(rows)


desc_train = calc_descriptors(train_mols)
desc_test = calc_descriptors(test_mols)
print("記述子の数:", desc_train.shape[1])


descriptors: 100%|██████████| 4000/4000 [00:12<00:00, 312.97it/s]


記述子の数: 217


In [5]:
MFP_RADIUS = 2
MFP_NBITS = 2048
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=MFP_RADIUS, fpSize=MFP_NBITS)


def calc_morgan(mols):
    """Morgan count fingerprint を (n_mol, NBITS) のDataFrameで返す。"""
    arr = np.zeros((len(mols), MFP_NBITS), dtype=np.float32)
    for i, m in enumerate(tqdm(mols, desc="morgan")):
        if m is not None:
            arr[i] = morgan_gen.GetCountFingerprintAsNumPy(m)
    cols = [f"mfp_{j}" for j in range(MFP_NBITS)]
    return pd.DataFrame(arr, columns=cols)


mfp_train = calc_morgan(train_mols)
mfp_test = calc_morgan(test_mols)
print("fingerprint 次元:", mfp_train.shape[1])


morgan: 100%|██████████| 4000/4000 [00:00<00:00, 34292.76it/s]

fingerprint 次元: 2048


## 5. 特徴量クリーニング

学習・推論で同じ処理を適用し、リークを防ぐ。

- `inf` を `NaN` に置換し、**trainの中央値**で欠損補完（同じ値を test にも適用）。
- trainで定数（分散ゼロ）の列を削除。
- test の列を train の列に完全一致させる。
- `float32` に統一。

記述子・fingerprint の両方に同じ関数を適用する。

In [6]:
def clean_features(train_feat, test_feat):
    """inf/NaN処理・定数列削除・train/test列整合を行う。統計量はtrainのみで算出（リーク防止）。"""
    train_feat = train_feat.replace([np.inf, -np.inf], np.nan)
    test_feat = test_feat.replace([np.inf, -np.inf], np.nan)

    medians = train_feat.median(numeric_only=True)          # trainの中央値で補完
    train_feat = train_feat.fillna(medians)
    test_feat = test_feat.fillna(medians)

    nunique = train_feat.nunique()                          # trainで定数の列を削除
    keep = nunique[nunique > 1].index
    train_feat = train_feat[keep]

    test_feat = test_feat.reindex(columns=keep, fill_value=0.0)  # 列を完全一致
    return train_feat.astype(np.float32), test_feat.astype(np.float32)


desc_train_c, desc_test_c = clean_features(desc_train, desc_test)
mfp_train_c, mfp_test_c = clean_features(mfp_train, mfp_test)
print("クリーニング後 — 記述子:", desc_train_c.shape[1], " / Morgan有効ビット:", mfp_train_c.shape[1])
assert list(desc_train_c.columns) == list(desc_test_c.columns)
assert list(mfp_train_c.columns) == list(mfp_test_c.columns)


クリーニング後 — 記述子: 187  / Morgan有効ビット: 2048


## 6. 交差検証の準備

全方針で **同一の fold**（`KFold(n_splits=5, shuffle=True, random_state=8)`）を使う。

`run_cv()` は各 fold で LightGBM を学習し、次を返す。
- OOF 予測（検証 fold の予測 → CV MAE の算出に使用）
- テスト予測（5つの fold モデルの平均）
- fold ごとの MAE

LightGBM は MAE を直接最適化する `regression_l1` を目的関数とし、
検証 fold で early stopping する。

In [7]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
y = train_df["gap_eV"].to_numpy(dtype=np.float64)

LGB_PARAMS = dict(
    objective="regression_l1",   # MAEを直接最適化
    metric="mae",
    learning_rate=0.03,
    num_leaves=127,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_estimators=5000,           # early stopping で実質的に打ち切られる
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)


def run_cv(X_train, X_test, y, params, name=""):
    """同一foldでLightGBMを学習し、OOF予測・テスト予測（fold平均）・fold別MAEを返す。"""
    Xtr = X_train.to_numpy(dtype=np.float32)
    Xte = X_test.to_numpy(dtype=np.float32)
    oof = np.zeros(len(y), dtype=np.float64)
    test_pred = np.zeros(Xte.shape[0], dtype=np.float64)
    fold_mae = []
    t0 = time.time()
    for fold, (tr, va) in enumerate(kf.split(Xtr)):
        model = lgb.LGBMRegressor(**params)
        model.fit(
            Xtr[tr], y[tr],
            eval_set=[(Xtr[va], y[va])],
            eval_metric="mae",
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
        )
        oof[va] = model.predict(Xtr[va])
        test_pred += model.predict(Xte) / N_SPLITS
        m = mean_absolute_error(y[va], oof[va])
        fold_mae.append(m)
        print(f"  [{name}] fold{fold}: MAE={m:.4f}  best_iter={model.best_iteration_}")
    cv = mean_absolute_error(y, oof)
    print(f"[{name}] CV MAE = {cv:.4f}  (fold平均 {np.mean(fold_mae):.4f} ± {np.std(fold_mae):.4f}, {time.time()-t0:.0f}s)")
    return oof, test_pred, fold_mae


## 7. 方針1 — RDKit記述子 → LightGBM

解釈しやすい標準2D記述子だけを使うベースの方針。

In [8]:
oof1, test1, mae1 = run_cv(desc_train_c, desc_test_c, y, LGB_PARAMS, name="方針1")


  [方針1] fold0: MAE=0.2235  best_iter=4999
  [方針1] fold1: MAE=0.2188  best_iter=5000
  [方針1] fold2: MAE=0.2131  best_iter=5000
  [方針1] fold3: MAE=0.2270  best_iter=5000
  [方針1] fold4: MAE=0.2151  best_iter=5000
[方針1] CV MAE = 0.2195  (fold平均 0.2195 ± 0.0051, 642s)


## 8. 方針2 — 記述子 ＋ Morgan → LightGBM

記述子に Morgan フィンガープリント（部分構造情報）を横結合して表現力を高める。
方針1から MAE が改善するかを確認する。

In [9]:
X2_train = pd.concat([desc_train_c, mfp_train_c], axis=1)
X2_test = pd.concat([desc_test_c, mfp_test_c], axis=1)
print("方針2の特徴量数:", X2_train.shape[1])
oof2, test2, mae2 = run_cv(X2_train, X2_test, y, LGB_PARAMS, name="方針2")


方針2の特徴量数: 2235
  [方針2] fold0: MAE=0.2154  best_iter=4996
  [方針2] fold1: MAE=0.2072  best_iter=4987
  [方針2] fold2: MAE=0.2049  best_iter=4296
  [方針2] fold3: MAE=0.2141  best_iter=5000
  [方針2] fold4: MAE=0.2076  best_iter=3918
[方針2] CV MAE = 0.2098  (fold平均 0.2098 ± 0.0041, 861s)


## 9. 方針3 — 方針1・2 のブレンド

方針1と方針2の予測を重み付き平均する。重み `w` は OOF 予測上で MAE を最小化する値を
0〜1 で簡易探索し、同じ重みをテスト予測にも適用する（予測の組み合わせはレギュレーション上可）。

In [10]:
best_w, best_mae = 0.5, np.inf
for w in np.linspace(0.0, 1.0, 21):
    m = mean_absolute_error(y, w * oof1 + (1.0 - w) * oof2)
    if m < best_mae:
        best_mae, best_w = m, w

oof3 = best_w * oof1 + (1.0 - best_w) * oof2
test3 = best_w * test1 + (1.0 - best_w) * test2
print(f"方針3: ブレンド重み w={best_w:.2f}（方針1の割合）, CV MAE={mean_absolute_error(y, oof3):.4f}")


方針3: ブレンド重み w=0.05（方針1の割合）, CV MAE=0.2098


## 10. CV 結果まとめ

3方針の CV MAE を一覧で比較する。

In [11]:
summary = pd.DataFrame(
    {
        "strategy": [
            "方針1: 記述子 → LGBM",
            "方針2: 記述子+Morgan → LGBM",
            f"方針3: ブレンド (w={best_w:.2f})",
        ],
        "cv_mae": [
            mean_absolute_error(y, oof1),
            mean_absolute_error(y, oof2),
            mean_absolute_error(y, oof3),
        ],
        "fold_std": [np.std(mae1), np.std(mae2), np.nan],
    }
)
display(summary)


,strategy,cv_mae,fold_std
0,方針1: 記述子 → LGBM,0.219519,0.005149
1,方針2: 記述子+Morgan → LGBM,0.209848,0.004117
2,方針3: ブレンド (w=0.05),0.209844,NaN


## 11. 提出ファイルの作成

各方針のテスト予測を `smiles,gap_eV`（列順固定・4000行）で書き出す。
書き出し前に、行数・列順・SMILES順序・欠損/無限値の有無を検証する。

In [12]:
def make_submission(pred, filename):
    """テストと同じ順序・行数で smiles,gap_eV のCSVを書き出し、妥当性を検証する。"""
    sub = pd.DataFrame({"smiles": test_df["smiles"].to_numpy(), "gap_eV": np.asarray(pred, dtype=np.float64)})
    assert len(sub) == len(test_df) == 4000, "行数が4000ではありません"
    assert list(sub.columns) == ["smiles", "gap_eV"], "列名/列順が不正です"
    assert sub["smiles"].tolist() == test_df["smiles"].tolist(), "SMILESの順序がテストと不一致"
    assert sub["gap_eV"].notna().all(), "欠損予測があります"
    assert np.isfinite(sub["gap_eV"].to_numpy()).all(), "無限値があります"
    sub.to_csv(filename, index=False)
    print(f"保存: {filename}  ({len(sub)}行)")
    return sub


s1 = make_submission(test1, "submission_1_descriptors_lgbm.csv")
s2 = make_submission(test2, "submission_2_descriptors_morgan_lgbm.csv")
s3 = make_submission(test3, "submission_3_blend.csv")
display(s1.head(3))


保存: submission_1_descriptors_lgbm.csv  (4000行)
保存: submission_2_descriptors_morgan_lgbm.csv  (4000行)
保存: submission_3_blend.csv  (4000行)


,smiles,gap_eV
0,CC1(CN1)C(C#C)C#N,7.270130
1,C1CC11C2OC2C11CO1,8.233562
2,OC1CC1C1CC(O)C1,8.624324


## 12. 用語メモ & 口頭説明のヒント

**用語**
- **QM9**: 小さな有機分子（最大9個の重原子: C,N,O,F）約13万件の量子化学計算データセット。
- **RDKit**: 分子の読み込み・特徴量計算ができるオープンソースのケモインフォマティクスライブラリ。
- **SMILES**: 分子構造を文字列で表す記法（例: `CCO` はエタノール）。
- **分子記述子**: 分子量や環の数など、分子を数値で特徴づける量。ここでは RDKit 標準2D記述子を利用。
- **フィンガープリント**: 部分構造の有無をビット列で表す特徴量。ここでは Morgan (ECFP相当) を利用。

**口頭説明の例（約1分）**
「SMILES から RDKit で標準2D記述子と Morgan フィンガープリントを生成し、5-fold CV の下で
LightGBM（MAE を目的関数）を学習しました。記述子のみ・記述子+FP・その2つのブレンドの3方針を比較し、
MAE が最も小さくなる構成を提出しています。」
